# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs"

feature_vector = pd.read_csv(f"{BASE}/feature_vector.csv")
labels = pd.read_csv(f"{BASE}/labels.csv")
ranked = pd.read_csv(f"{BASE}/ranked_baseline.csv")   # or 'scored' — whichever has the baseline score column

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from huggingface_hub import login

login()

In [5]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [6]:
from datasets import load_dataset
import duckdb


fact_content_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train")
fact_content = fact_content_ds.data.table   # Arrow, not pandas — much lighter

dim_content_ds = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train")
dim_content = dim_content_ds.data.table

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 4.41MB            

...

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [7]:
method_choice = "Logistic Regression (baseline model) -> Random Forest (complexity check)"
evaluation_metric = "precision@20, matching the Week-4 baseline's queue size"
print(method_choice)
print(evaluation_metric)

Logistic Regression (baseline model) -> Random Forest (complexity check)
precision@20, matching the Week-4 baseline's queue size


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [15]:
from sklearn.model_selection import GroupShuffleSplit

data_model = feature_vector.merge(labels, on=["client_hash_id", "content_hash_id"])
data_model = data_model.dropna(subset=["declined"])

honest_cols = ["word_count", "word_count_missing", "search_volume", "search_volume_missing",
               "competition_level_encoded", "content_age_days",
               "gsc_avg_position_prior", "gsc_position_missing"]

X = data_model[honest_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train clients:", data_model.iloc[train_idx]["client_hash_id"].nunique())
print("Test clients:", data_model.iloc[test_idx]["client_hash_id"].nunique())
print("Overlap check (should be 0):",
      len(set(data_model.iloc[train_idx]["client_hash_id"]) & set(data_model.iloc[test_idx]["client_hash_id"])))

Train clients: 28
Test clients: 13
Overlap check (should be 0): 0


In [14]:
feature_vector["search_volume_missing"] = feature_vector["search_volume"].isna().astype(int)
feature_vector["search_volume"] = feature_vector["search_volume"].fillna(feature_vector["search_volume"].median())

# confirm it's there now
print(feature_vector.columns.tolist())

['client_hash_id', 'content_hash_id', 'word_count', 'search_volume', 'competition_level', 'content_age_days', 'gsc_avg_position_prior', 'word_count_missing', 'gsc_position_missing', 'competition_level_encoded', 'search_volume_missing']


In [9]:
print(feature_vector.columns.tolist())

['client_hash_id', 'content_hash_id', 'word_count', 'search_volume', 'competition_level', 'content_age_days', 'gsc_avg_position_prior', 'word_count_missing', 'gsc_position_missing', 'competition_level_encoded']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [23]:
# === SECTION 3: Train + compare vs baseline ===

# --- Cell 1: build data_model with baseline score merged in (must happen before split) ---
import pandas as pd

baseline_df = pd.read_csv(f"{BASE}/ranked_baseline.csv")  # adjust filename if different

data_model = feature_vector.merge(labels, on=["client_hash_id", "content_hash_id"])
data_model = data_model.merge(
    baseline_df[["client_hash_id", "content_hash_id", "score"]].rename(columns={"score": "baseline_score"}),
    on=["client_hash_id", "content_hash_id"],
    how="left"
)
data_model = data_model.dropna(subset=["declined"])
data_model["baseline_score"] = data_model["baseline_score"].fillna(0)

print("data_model shape:", data_model.shape)

data_model shape: (134086, 15)


In [24]:
# --- Cell 2: grouped, honest split ---
from sklearn.model_selection import GroupShuffleSplit

honest_cols = ["word_count", "word_count_missing", "search_volume", "search_volume_missing",
               "competition_level_encoded", "content_age_days",
               "gsc_avg_position_prior", "gsc_position_missing"]

X = data_model[honest_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Client overlap (should be 0):",
      len(set(data_model.iloc[train_idx]["client_hash_id"]) & set(data_model.iloc[test_idx]["client_hash_id"])))

Train rows: 60528 | Test rows: 73558
Client overlap (should be 0): 0


In [34]:
X = data_model[honest_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(X_train.isna().sum())  # confirm all zero now

word_count                   0
word_count_missing           0
search_volume                0
search_volume_missing        0
competition_level_encoded    0
content_age_days             0
gsc_avg_position_prior       0
gsc_position_missing         0
dtype: int64


In [32]:
print(X_train.isna().sum())

word_count                      0
word_count_missing              0
search_volume                   0
search_volume_missing           0
competition_level_encoded    2535
content_age_days                0
gsc_avg_position_prior          0
gsc_position_missing            0
dtype: int64


In [28]:
print(data_model["competition_level"].unique())
print(data_model["competition_level"].isna().sum())

['LOW' 'HIGH' 'MEDIUM' nan]
11279


In [29]:
data_model["competition_level_encoded"] = data_model["competition_level_encoded"].fillna(-1)
print(data_model["competition_level_encoded"].isna().sum())  # should be 0

0


In [35]:
# --- Cell 3: train both models ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

print("Models trained successfully.")

Models trained successfully.


In [36]:
# --- Cell 4: precision@20 comparison table ---
import numpy as np

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    return y_true.iloc[top_k].mean() if hasattr(y_true, "iloc") else y_true[top_k].mean()

y_test_reset = y_test.reset_index(drop=True)
baseline_scores_test = data_model.iloc[test_idx]["baseline_score"].reset_index(drop=True)

base_rate = y_test_reset.mean()
baseline_p20 = precision_at_k(y_test_reset, baseline_scores_test.values, 20)
logreg_p20 = precision_at_k(y_test_reset, logreg_scores, 20)
rf_p20 = precision_at_k(y_test_reset, rf_scores, 20)

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "Week-4 rule baseline", "Logistic Regression", "Random Forest"],
    "precision@20": [base_rate, baseline_p20, logreg_p20, rf_p20]
})
comparison

,method,precision@20
0,Base rate (random),0.206912
1,Week-4 rule baseline,0.350000
2,Logistic Regression,0.450000
3,Random Forest,0.250000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [37]:
# === SECTION 4: Errors and interpretation ===

# --- Cell 5: feature importance ---
importances = pd.Series(rf.feature_importances_, index=honest_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

top_feature = importances.index[0]
top_share = importances.iloc[0]
print(f"\nTop feature: {top_feature} ({top_share:.1%} of total importance)")
print("⚠️ Check for leakage" if top_share > 0.7 else "✅ No single feature dominates")

Random Forest feature importances:
content_age_days             0.433973
word_count                   0.277049
gsc_avg_position_prior       0.136614
word_count_missing           0.090496
search_volume                0.033889
competition_level_encoded    0.027978
search_volume_missing        0.000000
gsc_position_missing         0.000000
dtype: float64

Top feature: content_age_days (43.4% of total importance)
✅ No single feature dominates


In [38]:
# --- Cell 6: build test_results for error analysis ---
test_results = data_model.iloc[test_idx].copy().reset_index(drop=True)
test_results["true_label"] = y_test_reset
test_results["rf_score"] = rf_scores
test_results["logreg_score"] = logreg_scores

false_negatives = test_results[test_results["true_label"] == 1].sort_values("rf_score").head(3)
print("Worst false negatives (missed real declines):")
print(false_negatives[["client_hash_id", "content_hash_id", "content_age_days",
                        "gsc_avg_position_prior", "rf_score"]])

Worst false negatives (missed real declines):
                client_hash_id           content_hash_id  content_age_days  \
33447  client_73cda7b4e4f265ea  content_8671edba7788e7ea               213   
28367  client_62f4a7e64f5e0096  content_0f067cb4ec91ab72               219   
66858  client_3f0ce4d44fe94f3d  content_3f230cbfac13abd8                17   

       gsc_avg_position_prior  rf_score  
33447               68.402990  0.117482  
28367               70.652702  0.117920  
66858              150.000000  0.123201  


In [39]:
# --- Cell 7: false positives ---
false_positives = test_results[test_results["true_label"] == 0].sort_values("rf_score", ascending=False).head(3)
print("Worst false positives (flagged but didn't decline):")
print(false_positives[["client_hash_id", "content_hash_id", "content_age_days",
                        "gsc_avg_position_prior", "rf_score"]])

Worst false positives (flagged but didn't decline):
                client_hash_id           content_hash_id  content_age_days  \
55052  client_fef1a8f436438636  content_eb127d86d0bc22cc               184   
51023  client_9958f0a7ae1df715  content_9804c8d434f01efd               417   
51255  client_9958f0a7ae1df715  content_ab7747aa4c264b41               417   

       gsc_avg_position_prior  rf_score  
55052               20.966169  0.693923  
51023                4.974344  0.693889  
51255                5.021106  0.693336  


In [40]:
# --- Cell 8: error patterns by group ---
test_results["age_bucket"] = pd.cut(test_results["content_age_days"],
                                      bins=[-1, 90, 180, 10000], labels=["<90d", "90-180d", "180d+"])
test_results["correct"] = (test_results["rf_score"] > 0.5) == (test_results["true_label"] == 1)

print("Accuracy by age bucket:")
print(test_results.groupby("age_bucket")["correct"].mean())

print("\nAccuracy by whether gsc_avg_position_prior was missing:")
print(test_results.groupby("gsc_position_missing")["correct"].mean())

Accuracy by age bucket:
age_bucket
<90d       0.822845
90-180d    0.688128
180d+      0.712354
Name: correct, dtype: float64

Accuracy by whether gsc_avg_position_prior was missing:
gsc_position_missing
0    0.740123
Name: correct, dtype: float64


/tmp/ipykernel_3176/3170365085.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(test_results.groupby("age_bucket")["correct"].mean())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.